# 07 — DQN Training (Tutorial-Inspired, SB3)

DQN agent on Super Mario Bros using SB3, with hyperparameters from the
[PyTorch Mario RL tutorial](https://docs.pytorch.org/tutorials/intermediate/mario_rl_tutorial.html).

**Tutorial params adapted to SB3:**
- Same CNN architecture (Nature CNN = tutorial's MarioNet)
- gamma=0.9 (short-term focus, from tutorial)
- lr=0.00025 (from tutorial)
- Epsilon: 1.0 → 0.1 over 10% of training
- buffer_size=100,000, batch_size=32
- target_update_interval=10,000
- train_freq=3 (learn_every=3 from tutorial)
- total_timesteps=2,000,000
- Checkpoints every 100k steps + TensorBoard

In [ ]:
# --- Google Colab Setup ---
import os, sys

if os.getenv('COLAB_RELEASE_TAG'):
    !pip install -q Cython setuptools wheel
    !git clone -b hotfix/version1 https://github.com/lmartim4/CSC-52081-ContinousMountainCar.git /content/repo
    %cd /content/repo
    !pip install -r requirements.txt --no-build-isolation
    !pip install tensordict torchrl
    sys.path.insert(0, '/content/repo')
    import site; SITE = site.getsitepackages()[0]
    !patch --forward -p0 {SITE}/nes_py/_rom.py                   < patches/nes_py_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym/utils/passive_env_checker.py < patches/gym_bool8_numpy2.patch || true
    !patch --forward -p0 {SITE}/gym_super_mario_bros/smb_env.py  < patches/smb_env_numpy2.patch || true
    !sed -i 's/observation, reward, terminated, truncated, info = self.env.step(action)/_result = self.env.step(action); observation, reward, terminated, info = _result[:4]; truncated = _result[4] if len(_result) > 4 else False/' {SITE}/gym/wrappers/time_limit.py
    !git pull
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        %cd ..

In [ ]:
import torch
from stable_baselines3 import DQN
from stable_baselines3.common.monitor import Monitor

from src.wrappers import make_pixel_env
from src.utils.callbacks import CheckpointAndLogCallback
from src.config import DQNConfig

print(f'Using CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Create environment (v3 has built-in time limit)
env = make_pixel_env(env_id='SuperMarioBros-1-1-v3')
env = Monitor(env)
print(f'Observation space: {env.observation_space.shape}')
print(f'Action space: {env.action_space.n}')

In [ ]:
# Tutorial-inspired hyperparameters via SB3 DQN
# SB3 CnnPolicy uses the same Nature CNN as the tutorial's MarioNet:
#   Conv2d(c,32,8,4) -> Conv2d(32,64,4,2) -> Conv2d(64,64,3,1) -> Linear(3136,512)

config = DQNConfig(
    policy='CnnPolicy',
    learning_rate=0.00025,         # from tutorial
    gamma=0.9,                     # from tutorial (short-term focus)
    buffer_size=100_000,           # from tutorial
    batch_size=32,                 # from tutorial
    learning_starts=10_000,        # burnin from tutorial
    target_update_interval=10_000, # sync_every from tutorial
    train_freq=3,                  # learn_every from tutorial
    exploration_fraction=0.10,     # decay epsilon over 10% of training
    exploration_initial_eps=1.0,   # from tutorial
    exploration_final_eps=0.1,     # from tutorial (not 0.01)
    total_timesteps=2_000_000,
)

model = DQN(
    policy=config.policy,
    env=env,
    learning_rate=config.learning_rate,
    buffer_size=config.buffer_size,
    learning_starts=config.learning_starts,
    batch_size=config.batch_size,
    gamma=config.gamma,
    target_update_interval=config.target_update_interval,
    exploration_fraction=config.exploration_fraction,
    exploration_initial_eps=config.exploration_initial_eps,
    exploration_final_eps=config.exploration_final_eps,
    train_freq=config.train_freq,
    tensorboard_log='../logs/tutorial_dqn',
    verbose=1,
)

print(f'Total timesteps: {config.total_timesteps:,}')
print(f'Device: {model.device}')

In [ ]:
%load_ext tensorboard
%tensorboard --logdir ../logs/tutorial_dqn

In [ ]:
# Train with checkpoints
callback = CheckpointAndLogCallback(
    save_path='../models/tutorial_dqn',
    save_freq=100_000,   # checkpoint every 100k steps
)

model.learn(
    total_timesteps=config.total_timesteps,
    callback=callback,
    log_interval=100,  # log to tensorboard every 100 episodes
)
model.save('../models/tutorial_dqn/final_model')
print('Training complete! Final model saved.')

In [ ]:
# Resume from checkpoint (if training was interrupted)
# Uncomment to load a checkpoint and continue training:

# model = DQN.load('../models/tutorial_dqn/model_100000', env=env)
# model.learn(total_timesteps=remaining_steps, callback=callback, reset_num_timesteps=False)

In [ ]:
# Plot training results from callback
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

if len(callback.episode_rewards) > 0:
    window = min(100, len(callback.episode_rewards))

    # Rewards
    ax = axes[0]
    ax.plot(callback.episode_rewards, alpha=0.3, color='blue')
    if len(callback.episode_rewards) > window:
        smoothed = np.convolve(callback.episode_rewards, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(callback.episode_rewards)), smoothed, color='blue', linewidth=2)
    ax.set_title('Episode Rewards')
    ax.set_xlabel('Episode')

    # Lengths
    ax = axes[1]
    ax.plot(callback.episode_lengths, alpha=0.3, color='orange')
    if len(callback.episode_lengths) > window:
        smoothed = np.convolve(callback.episode_lengths, np.ones(window)/window, mode='valid')
        ax.plot(range(window-1, len(callback.episode_lengths)), smoothed, color='orange', linewidth=2)
    ax.set_title('Episode Lengths')
    ax.set_xlabel('Episode')

    # Flag rate
    ax = axes[2]
    if len(callback.episode_flags) > 0:
        cumulative_flags = np.cumsum(callback.episode_flags) / (np.arange(len(callback.episode_flags)) + 1)
        ax.plot(cumulative_flags, color='green', linewidth=2)
    ax.set_title('Cumulative Flag Rate')
    ax.set_xlabel('Episode')
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('../models/tutorial_dqn/training_curves.png', dpi=150)
plt.show()
else:
    print('No episode data collected yet.')

In [ ]:
# Evaluate the trained model
from src.utils.evaluation import evaluate_agent

eval_env = make_pixel_env(env_id='SuperMarioBros-1-1-v3')
results = evaluate_agent(model, eval_env, n_episodes=30)
print(f"Mean reward: {results['mean_reward']:.1f} +/- {results['std_reward']:.1f}")
print(f"Flag rate: {results['flag_rate']:.2%}")